# Cybersecurity Data Pipeline

Pipeline:
- Data collection (logs + API)
- Analysis
- Response
- Reporting


## 1. Imports

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import json

## 2. Simulated Log Collection (Suricata)

In [ ]:
logs = [
    {"ip": "1.2.3.4", "event": "dns"},
    {"ip": "1.2.3.4", "event": "dns"},
    {"ip": "5.6.7.8", "event": "http"},
    {"ip": "1.2.3.4", "event": "dns"},
    {"ip": "9.9.9.9", "event": "dns"},
]

df_logs = pd.DataFrame(logs)
df_logs

## 3. Analyze Suspicious IPs

In [ ]:
ip_counts = df_logs["ip"].value_counts()
suspicious_ips = ip_counts[ip_counts > 2]

suspicious_ips

## 4. Vulnerability API (Vulners)

In [ ]:
def get_cvss(cve):
    url = "https://vulners.com/api/v3/search/id/"
    params = {"id": cve}
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json()
        return data.get("data", {}).get("documents", {}).get(cve, {}).get("cvss", {}).get("score", 0)
    except:
        return 0

cve_list = ["CVE-2021-44228", "CVE-2020-1472"]

vuln_data = []
for cve in cve_list:
    score = get_cvss(cve)
    vuln_data.append({"cve": cve, "cvss": score})

df_vuln = pd.DataFrame(vuln_data)
df_vuln

## 5. Detect High-Risk Vulnerabilities

In [ ]:
high_risk = df_vuln[df_vuln["cvss"] > 7]
high_risk

## 6. Response Simulation

In [ ]:
print("=== RESPONSE ===")

for ip in suspicious_ips.index:
    print(f"[BLOCK] Suspicious IP: {ip}")

for _, row in high_risk.iterrows():
    print(f"[ALERT] High CVE: {row['cve']} (CVSS={row['cvss']})")

## 7. Reporting

In [ ]:
report = {
    "suspicious_ips": suspicious_ips.to_dict(),
    "high_risk_cve": high_risk.to_dict(orient="records"),
}

with open("report.json", "w") as f:
    json.dump(report, f, indent=4)

df_vuln.to_csv("report.csv", index=False)

print("Reports saved")

## 8. Visualization

In [ ]:
df_logs["ip"].value_counts().plot(kind="bar")
plt.title("Top IPs")
plt.savefig("plot.png")
plt.show()